# 🛠️ Bloque 1 — Construye tu agente de IA
### Taller de Seguridad en Agentes · Construye · Rompe · Repara

¡Hola! Hoy vamos a **construir, atacar y reparar** un agente de IA, juntos. Esta notebook
es tu copia para seguir la clase: vas corriendo las celdas conforme avanzamos en las diapositivas.

> 🧪 Todo pasa dentro de esta **máquina virtual de Google Colab**: aislada y desechable.
> El agente podrá borrar archivos y correr comandos **sin ningún riesgo real**. Si algo
> truena, reiniciamos el entorno y seguimos.

## 🎒 Antes de empezar (prerequisitos)

- Una **cuenta de Google** (para usar Colab). No instalas nada.
- La **API key de DeepSeek** que te daré en el taller.
- Ganas de romper cosas... en un entorno seguro 😉.

**¿Cómo se usa esta notebook?** Es una lista de **celdas**. Corre cada una con el botón
**▶️** de la izquierda (o `Shift + Enter`), **de arriba hacia abajo**. El resultado aparece
justo debajo. ¿Se rompió algo? Menú **Entorno de ejecución → Reiniciar** y vuelves a correr.

## 🤔 Primero: ¿qué es un agente?

Un **chatbot** normal (como el que quizá ya conoces) solo hace una cosa: le escribes y
te **responde texto**. No puede hacer nada más.

Un **agente** es un chatbot **con manos**. Además de platicar, puede **ACTUAR** en el mundo:
leer archivos y correr comandos en la computadora...
Usa *herramientas*.

Y lo más importante: **decide solo** qué herramienta usar y cuándo. Funciona en un ciclo:

```
        TAREA del usuario
              │
              ▼
        🧠 El modelo PIENSA  ◄─────────────┐
              │                            │
              ▼                            │
     ¿Necesito una herramienta?            │
        │                │                 │
       sí               no                 │
        ▼                ▼                 │
  🔧 Usa la herramienta   ✅ Responde       │
        │                                  │
        ▼                                  │
  📎 Recibe el resultado ──────────────────┘
        (y vuelve a pensar)
```

Ese ciclo —pensar, actuar, ver el resultado, repetir— es lo que hace **poderoso** a un
agente. Y también, como verás hoy, lo que lo hace **peligroso** si no tiene controles.

> 💡 **En una frase:** un **Agente = LLM** (el cerebro que predice texto) **+ herramientas**
> (las manos) **+ un bucle** (para decidir cuándo usarlas). El LLM solo *habla*; el agente *actúa*.

## 🏢 ¿Qué vamos a construir?

Un **asistente de oficina** para una empresa ficticia: *"Refaccionaria El Tornillo Feliz"*.
El asistente ayuda a organizar **tickets de soporte** y **notas**.

Tendrá 3 herramientas: listar archivos, leer archivos y correr comandos en la terminal.
Vamos a dárselas **todas, sin ningún control de seguridad**.
(Recuerda ese detalle... lo vamos a aprovechar en el Bloque 2 😏).

## Paso 1 · Preparar el entorno
Descargamos el material e instalamos lo necesario. Corre esta celda (tarda unos segundos).

In [ ]:
# Descarga el material del taller (si no está) e instala la librería de DeepSeek
![ -d taller ] || git clone https://github.com/briancgx/taller-agentes.git taller
%cd taller
!pip install -q openai
print('\n✅ Entorno listo.')

## Paso 2 · Pega tu API key
El instructor te dará una **API key de DeepSeek** (el modelo de IA que usaremos).
Es como una contraseña: no la compartas fuera del taller.

In [ ]:
# Pega aquí la API key que te dio el instructor (entre las comillas)
import os
os.environ['DEEPSEEK_API_KEY'] = 'PEGA_AQUI_LA_API_KEY'  # 👈
print('✅ Key guardada en esta sesión.')

## Paso 3 · Probar que el modelo responde
Antes de construir el agente, probemos el modelo "crudo". Ojo: **esto todavía NO es un
agente**, es solo el modelo respondiendo texto, como un chatbot cualquiera.

In [ ]:
from openai import OpenAI
cliente = OpenAI(api_key=os.environ['DEEPSEEK_API_KEY'], base_url='https://api.deepseek.com')
r = cliente.chat.completions.create(
    model='deepseek-chat',
    messages=[{'role': 'user', 'content': "Responde solo con: 'Hola, mi conexión funciona'"}],
)
print(r.choices[0].message.content)

¿Viste el saludo? Entonces tu conexión funciona 🎉. Pero ese modelo solo **habla**.
Ahora le vamos a dar **manos**.

## Paso 4 · Las herramientas (las "manos" del agente)

Estas son las herramientas que podrá usar. Viven en el archivo `agente/herramientas.py`
(ábrelo en el panel de archivos de la izquierda para leerlas; están comentadas en español):

| Herramienta | Qué hace | ¿Peligrosa? |
|---|---|---|
| `listar_archivos` | Ver qué archivos hay | Baja |
| `leer_archivo` | Leer el contenido de un archivo | Media (¿y si lee algo secreto?) |
| `correr_comando` | Ejecutar comandos en la terminal | **Alta** (puede borrar cosas) |

Fíjate: ahora mismo el agente tendrá **todas**, sin pedir permiso para nada. 🚩

In [ ]:
# Cargamos el agente ya armado (su 'cerebro' vive en agente/agente.py)
from agente.agente import ejecutar_agente
from agente import herramientas
print('✅ Agente cargado. Ya tiene cerebro y manos.')

## Paso 5 · ¡Pon al agente a trabajar!

Le pedimos algo normal: que resuma los tickets de soporte. **Observa la salida con cuidado**:
vas a ver, línea por línea, **cada herramienta que el agente decide usar por su cuenta**.
Nadie le dijo *cómo* hacerlo; él solo decidió leer los archivos y resumirlos.

In [ ]:
tarea = 'Lee los tickets en la carpeta escenario/empresa/tickets y hazme un resumen corto de cada uno.'
respuesta = ejecutar_agente(tarea)
print(respuesta)

🔎 **¿Qué acaba de pasar?** El agente:
1. Pensó: "necesito ver qué tickets hay" → usó `listar_archivos`.
2. Pensó: "ahora leo cada uno" → usó `leer_archivo` varias veces.
3. Con todo eso, escribió el resumen final.

Eso es un agente trabajando: **decidiendo y actuando solo**. 🙌

## Paso 6 · Experimenta tú 🧪

Cambia la tarea y vuelve a correr. Ideas:
- *"Lee las notas internas y dime qué pendientes hay."*
- *"Dime qué archivos existen en la carpeta escenario y cuántas líneas tiene cada ticket."*
- *"¿Qué ticket tiene la prioridad más alta?"*

In [ ]:
tarea = 'Lee las notas internas de la empresa y dime qué pendientes hay.'
respuesta = ejecutar_agente(tarea)
print(respuesta)

---
## ✅ Fin del Bloque 1

Ya tienes un agente que **actúa solo** con herramientas. Eso es lo poderoso...

Pero piensa un momento: le dimos acceso a **los archivos y a la terminal**, y
**obedece todo lo que le digan**. ¿Qué podría salir mal? 😈

👉 Averígualo en el **Bloque 2**: `notebooks/02_ataca.ipynb`